In [2]:
!pip install gradio deep-translator gTTS langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 35.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of typer to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.3/122.3 kB 10.6 MB/s eta 0:00:00
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=8c1fb55ad92bdc7069b91df1c093d6d68cc20d2f46f94722f4d334e5448f1d43
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect
  Attempting uninstall: click
    Found existing installation: click 8.4.0
    Uninstalling click-8.4.0:
      Successfully uninstalled click-8.4.0
  Attempting uninstall: typer
    Found existing instal

In [3]:
import gradio as gr
from deep_translator import GoogleTranslator
from gtts import gTTS
from langdetect import detect
import tempfile
import os

In [4]:
LANGUAGES = {
    "🇵🇰 Urdu":           {"code": "ur",    "tts": "ur",    "tld": "com"},
    "🇸🇦 Arabic":          {"code": "ar",    "tts": "ar",    "tld": "com"},
    "🇫🇷 French":          {"code": "fr",    "tts": "fr",    "tld": "fr"},
    "🇪🇸 Spanish":         {"code": "es",    "tts": "es",    "tld": "es"},
    "🇨🇳 Chinese":         {"code": "zh-CN", "tts": "zh-CN", "tld": "com"},
    "🇮🇳 Hindi":           {"code": "hi",    "tts": "hi",    "tld": "com"},
    "🇩🇪 German":          {"code": "de",    "tts": "de",    "tld": "de"},
    "🇯🇵 Japanese":        {"code": "ja",    "tts": "ja",    "tld": "com"},
    "🇹🇷 Turkish":         {"code": "tr",    "tts": "tr",    "tld": "com"},
    "🇷🇺 Russian":         {"code": "ru",    "tts": "ru",    "tld": "com"},
    "🇬🇧 English (UK)":   {"code": "en",    "tts": "en",    "tld": "co.uk"},
    "🇺🇸 English (US)":   {"code": "en",    "tts": "en",    "tld": "com"},
    "🇧🇷 Portuguese":      {"code": "pt",    "tts": "pt",    "tld": "com.br"},
    "🇮🇹 Italian":         {"code": "it",    "tts": "it",    "tld": "it"},
    "🇰🇷 Korean":          {"code": "ko",    "tts": "ko",    "tld": "com"},
    "🇳🇱 Dutch":           {"code": "nl",    "tts": "nl",    "tld": "com"},
    "🇸🇪 Swedish":         {"code": "sv",    "tts": "sv",    "tld": "com"},
    "🇵🇱 Polish":          {"code": "pl",    "tts": "pl",    "tld": "com"},
    "🇮🇩 Indonesian":      {"code": "id",    "tts": "id",    "tld": "com"},
    "🇹🇭 Thai":            {"code": "th",    "tts": "th",    "tld": "com"},
}

LANG_NAMES = list(LANGUAGES.keys())

print(f"✅ {len(LANGUAGES)} languages loaded!")

✅ 20 languages loaded!


In [5]:
def translate_and_speak(text, src_lang_name, tgt_lang_name, auto_detect, slow_speech):
    if not text or not text.strip():
        return "⚠️ Please enter some text.", None, "—", "—"

    src_info = LANGUAGES.get(src_lang_name, {})
    tgt_info = LANGUAGES.get(tgt_lang_name, {})

    # Source language
    if auto_detect:
        try:
            detected = detect(text)
            src_code = detected
            detected_name = next(
                (k for k, v in LANGUAGES.items() if v["code"] == detected), detected
            )
            src_label = f"🔍 Auto-detected: {detected_name}"
        except:
            src_code = "auto"
            src_label = "🔍 Auto-detect (fallback)"
    else:
        src_code  = src_info.get("code", "auto")
        src_label = f"📌 {src_lang_name}"

    tgt_code = tgt_info.get("code", "en")
    tts_lang = tgt_info.get("tts", "en")
    tts_tld  = tgt_info.get("tld", "com")

    # Translate
    try:
        translated = GoogleTranslator(source=src_code, target=tgt_code).translate(text)
    except Exception as e:
        return f"❌ Translation failed: {e}", None, src_label, "—"

    # TTS
    try:
        tts = gTTS(text=translated, lang=tts_lang, tld=tts_tld, slow=slow_speech)
        tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3")
        tts.save(tmp.name)
        audio_path = tmp.name
    except Exception as e:
        audio_path = None

    char_info = f"📊 {len(text)} chars input → {len(translated)} chars output"
    return translated, audio_path, src_label, char_info

print("✅ Function ready!")

✅ Function ready!


In [22]:
CSS = """
@import url('https://fonts.googleapis.com/css2?family=Fredoka+One&family=Nunito:wght@300;400;500;600;700;800&display=swap');

* { box-sizing: border-box; margin: 0; padding: 0; }

body, .gradio-container {
    font-family: 'Nunito', sans-serif !important;
    background: #fef6ff !important;
}

.gradio-container {
    max-width: 1000px !important;
    margin: 0 auto !important;
    padding: 1.5rem !important;
}

/* ── Animated background blobs ── */
.gradio-container::before {
    content: '';
    position: fixed;
    top: -100px; left: -100px;
    width: 400px; height: 400px;
    background: radial-gradient(circle, #f9a8d4 0%, transparent 70%);
    border-radius: 50%;
    animation: blob1 8s ease-in-out infinite;
    pointer-events: none;
    z-index: 0;
}
.gradio-container::after {
    content: '';
    position: fixed;
    bottom: -100px; right: -100px;
    width: 350px; height: 350px;
    background: radial-gradient(circle, #a5b4fc 0%, transparent 70%);
    border-radius: 50%;
    animation: blob2 10s ease-in-out infinite;
    pointer-events: none;
    z-index: 0;
}
@keyframes blob1 {
    0%,100% { transform: translate(0,0) scale(1); }
    33%      { transform: translate(40px,30px) scale(1.1); }
    66%      { transform: translate(-20px,50px) scale(0.95); }
}
@keyframes blob2 {
    0%,100% { transform: translate(0,0) scale(1); }
    33%      { transform: translate(-30px,-40px) scale(1.08); }
    66%      { transform: translate(20px,-20px) scale(0.97); }
}

/* ── Hero header ── */
.hero-wrap {
    text-align: center;
    padding: 2rem 1rem 1.5rem;
    position: relative;
    z-index: 1;
}
.mascot {
    font-size: 4rem;
    animation: bounce 2.5s ease-in-out infinite;
    display: block;
    line-height: 1;
    margin-bottom: 0.5rem;
}
@keyframes bounce {
    0%,100% { transform: translateY(0); }
    50%      { transform: translateY(-10px); }
}
.app-title {
    font-family: 'Fredoka One', cursive !important;
    font-size: 2.8rem !important;
    background: linear-gradient(135deg, #ec4899, #8b5cf6, #3b82f6);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    background-clip: text;
    letter-spacing: 0.02em;
    line-height: 1;
    margin-bottom: 0.4rem !important;
}
.app-sub {
    color: #a78bfa !important;
    font-size: 0.95rem !important;
    font-weight: 600 !important;
    letter-spacing: 0.06em;
}
.badge-row {
    display: flex;
    gap: 8px;
    justify-content: center;
    flex-wrap: wrap;
    margin-top: 0.8rem;
}
.badge {
    background: white;
    border: 1.5px solid #e9d5ff;
    border-radius: 999px;
    padding: 4px 12px;
    font-size: 0.72rem;
    font-weight: 700;
    color: #7c3aed;
    letter-spacing: 0.05em;
}

/* ── Cards ── */
.card {
    background: white;
    border: 2px solid #f3e8ff;
    border-radius: 24px;
    padding: 1.4rem 1.4rem 1.2rem;
    position: relative;
    z-index: 1;
    box-shadow: 0 4px 24px rgba(167,139,250,0.10);
    transition: box-shadow 0.2s;
}
.card:hover {
    box-shadow: 0 8px 32px rgba(167,139,250,0.18);
}
.card-title {
    font-family: 'Fredoka One', cursive;
    font-size: 1rem;
    color: #7c3aed;
    margin-bottom: 1rem;
    display: flex;
    align-items: center;
    gap: 6px;
    letter-spacing: 0.03em;
}

/* ── Inputs ── */
.gradio-container textarea,
.gradio-container input[type="text"] {
    background: #faf5ff !important;
    border: 2px solid #e9d5ff !important;
    border-radius: 16px !important;
    color: #3b0764 !important;
    font-family: 'Nunito', sans-serif !important;
    font-size: 1rem !important;
    font-weight: 500 !important;
    padding: 0.9rem 1rem !important;
    transition: border-color 0.2s, box-shadow 0.2s !important;
    line-height: 1.6 !important;
}
.gradio-container textarea:focus,
.gradio-container input[type="text"]:focus {
    border-color: #a855f7 !important;
    box-shadow: 0 0 0 4px rgba(168,85,247,0.12) !important;
    outline: none !important;
}

/* ── Labels ── */
.gradio-container label span,
.gradio-container .label-wrap span {
    color: #9333ea !important;
    font-size: 0.8rem !important;
    font-weight: 700 !important;
    letter-spacing: 0.06em !important;
    text-transform: uppercase !important;
}

/* ── Dropdowns ── */
.gradio-container select {
    background: #faf5ff !important;
    border: 2px solid #e9d5ff !important;
    border-radius: 14px !important;
    color: #3b0764 !important;
    font-family: 'Nunito', sans-serif !important;
    font-weight: 600 !important;
    padding: 0.6rem 1rem !important;
}

/* ── Translate button ── */
.go-btn {
    background: linear-gradient(135deg, #ec4899 0%, #8b5cf6 60%, #3b82f6 100%) !important;
    border: none !important;
    border-radius: 16px !important;
    color: white !important;
    font-family: 'Fredoka One', cursive !important;
    font-size: 1.15rem !important;
    letter-spacing: 0.05em !important;
    padding: 0.8rem 1.5rem !important;
    width: 100% !important;
    cursor: pointer !important;
    transition: transform 0.18s, box-shadow 0.18s !important;
    box-shadow: 0 4px 18px rgba(139,92,246,0.35) !important;
}
.go-btn:hover {
    transform: translateY(-3px) scale(1.02) !important;
    box-shadow: 0 10px 28px rgba(139,92,246,0.45) !important;
}
.go-btn:active {
    transform: scale(0.98) !important;
}

/* ── Utility buttons ── */
.swap-btn {
    background: #f0fdf4 !important;
    border: 2px solid #bbf7d0 !important;
    border-radius: 12px !important;
    color: #16a34a !important;
    font-family: 'Nunito', sans-serif !important;
    font-size: 0.85rem !important;
    font-weight: 700 !important;
    cursor: pointer !important;
    transition: all 0.18s !important;
    padding: 0.45rem 0.8rem !important;
}
.swap-btn:hover {
    background: #dcfce7 !important;
    transform: rotate(180deg) !important;
}
.clear-btn {
    background: #fff1f2 !important;
    border: 2px solid #fecdd3 !important;
    border-radius: 12px !important;
    color: #e11d48 !important;
    font-family: 'Nunito', sans-serif !important;
    font-size: 0.85rem !important;
    font-weight: 700 !important;
    cursor: pointer !important;
    transition: all 0.18s !important;
    padding: 0.45rem 0.8rem !important;
}
.clear-btn:hover { background: #ffe4e6 !important; }

/* ── Output textarea ── */
.output-box textarea {
    border-color: #bfdbfe !important;
    background: #eff6ff !important;
    color: #1e40af !important;
    font-size: 1.05rem !important;
    font-weight: 600 !important;
}

/* ── Stat chips ── */
.gradio-container .stat-chip textarea,
.gradio-container .stat-chip input {
    background: #fdf4ff !important;
    border: 2px solid #f3e8ff !important;
    border-radius: 12px !important;
    color: #7c3aed !important;
    font-size: 0.8rem !important;
    font-weight: 700 !important;
    text-align: center !important;
    padding: 0.4rem 0.6rem !important;
}

/* ── Audio ── */
.gradio-container audio {
    width: 100% !important;
    border-radius: 14px !important;
    background: #faf5ff !important;
    border: 2px solid #e9d5ff !important;
}

/* ── Checkbox ── */
.gradio-container input[type="checkbox"] {
    accent-color: #a855f7 !important;
    width: 16px !important;
    height: 16px !important;
}
.gradio-container .wrap.svelte-1p9xokt { gap: 8px !important; }

/* ── Examples ── */
.gradio-container .examples-header {
    font-family: 'Fredoka One', cursive !important;
    color: #7c3aed !important;
    font-size: 1rem !important;
    letter-spacing: 0.03em !important;
}
.gradio-container table.examples {
    background: white !important;
    border: 2px solid #f3e8ff !important;
    border-radius: 16px !important;
    overflow: hidden !important;
}
.gradio-container table.examples td {
    color: #6b21a8 !important;
    font-size: 0.85rem !important;
    font-weight: 600 !important;
    border-color: #f3e8ff !important;
    cursor: pointer !important;
    transition: background 0.15s !important;
}
.gradio-container table.examples tr:hover td {
    background: #fdf4ff !important;
}

/* ── Divider ── */
.divider { border: none; border-top: 2px dashed #f3e8ff; margin: 1rem 0; }

/* ── Floating stars ── */
.stars {
    position: fixed; top: 0; left: 0;
    width: 100%; height: 100%;
    pointer-events: none; z-index: 0;
    overflow: hidden;
}
.star {
    position: absolute;
    font-size: 1.2rem;
    animation: floatStar linear infinite;
    opacity: 0.5;
}
@keyframes floatStar {
    0%   { transform: translateY(110vh) rotate(0deg); opacity: 0; }
    10%  { opacity: 0.6; }
    90%  { opacity: 0.6; }
    100% { transform: translateY(-10vh) rotate(360deg); opacity: 0; }
}
"""

HEADER_HTML = """
<div class="stars">
  <span class="star" style="left:8%;  animation-duration:12s; animation-delay:0s;   font-size:1rem;">✨</span>
  <span class="star" style="left:20%; animation-duration:15s; animation-delay:3s;   font-size:0.8rem;">🌸</span>
  <span class="star" style="left:35%; animation-duration:11s; animation-delay:1s;   font-size:1.1rem;">⭐</span>
  <span class="star" style="left:55%; animation-duration:14s; animation-delay:5s;   font-size:0.9rem;">✨</span>
  <span class="star" style="left:70%; animation-duration:13s; animation-delay:2s;   font-size:1rem;">🌺</span>
  <span class="star" style="left:85%; animation-duration:16s; animation-delay:4s;   font-size:0.8rem;">⭐</span>
  <span class="star" style="left:92%; animation-duration:10s; animation-delay:6s;   font-size:1rem;">✨</span>
</div>

<div class="hero-wrap">
  <span class="mascot">🦜</span>
  <div class="app-title">LinguaBird</div>
  <div class="app-sub">Your cute multilingual translator bestie ✨</div>
  <div class="badge-row">
    <span class="badge">🌍 20 LANGUAGES</span>
    <span class="badge">🔊 REAL-TIME TTS</span>
    <span class="badge">🇬🇧 BRITISH ACCENT</span>
    <span class="badge">🤖 AUTO-DETECT</span>
  </div>
</div>
"""

with gr.Blocks(css=CSS, title="LinguaBird 🦜") as demo:

    gr.HTML(HEADER_HTML)

    with gr.Row(equal_height=False):

        # ── LEFT: Input panel ──────────────────────────────────────────────
        with gr.Column(scale=1):
            with gr.Group(elem_classes=["card"]):
                gr.HTML('<div class="card-title">🎤 What do you want to say?</div>')

                input_text = gr.Textbox(
                    label="Type your text here",
                    placeholder="میڈم میرہا آپ ہمیں پریکٹیکل میں پورے نمبر دیں۔  🙏",
                    lines=6,
                    max_lines=10,
                    show_copy_button=True,
                )

                with gr.Row():
                    auto_detect = gr.Checkbox(label="🔍 Auto-detect", value=True, scale=1)
                    slow_speech = gr.Checkbox(label="🐢 Slow speech", value=False, scale=1)

                src_lang = gr.Dropdown(
                    choices=LANG_NAMES,
                    value="🇵🇰 Urdu",
                    label="📌 Speak in",
                    interactive=True,
                )
                tgt_lang = gr.Dropdown(
                    choices=LANG_NAMES,
                    value="🇬🇧 English (UK)",
                    label="🎯 Translate to",
                    interactive=True,
                )

                with gr.Row():
                    swap_btn  = gr.Button("⇄ Swap",  elem_classes=["swap-btn"],  scale=1)
                    clear_btn = gr.Button("🗑 Clear", elem_classes=["clear-btn"], scale=1)

                gr.HTML('<hr class="divider">')
                translate_btn = gr.Button(
                    "🦜 Translate & Speak!",
                    elem_classes=["go-btn"],
                    variant="primary"
                )

        # ── RIGHT: Output panel ────────────────────────────────────────────
        with gr.Column(scale=1):
            with gr.Group(elem_classes=["card"]):
                gr.HTML('<div class="card-title">💬 Here\'s your translation!</div>')

                output_text = gr.Textbox(
                    label="Translation",
                    lines=6,
                    max_lines=10,
                    interactive=False,
                    show_copy_button=True,
                    elem_classes=["output-box"],
                )

                audio_out = gr.Audio(
                    label="🔊 Listen now",
                    type="filepath",
                    interactive=False,
                    autoplay=True,
                )

                with gr.Row():
                    src_info_box  = gr.Textbox(
                        label="🔍 Detected",
                        interactive=False,
                        scale=1,
                        elem_classes=["stat-chip"],
                    )
                    char_info_box = gr.Textbox(
                        label="📊 Stats",
                        interactive=False,
                        scale=1,
                        elem_classes=["stat-chip"],
                    )

    # ── Examples ──────────────────────────────────────────────────────────
    gr.HTML('<br>')
    with gr.Group(elem_classes=["card"]):
        gr.HTML('<div class="card-title">🌟 Try these examples!</div>')
        gr.Examples(
            examples=[
                ["مصنوعی ذہانت دنیا کو بدل رہی ہے۔",              "🇵🇰 Urdu",        "🇬🇧 English (UK)", True,  False],
                ["مرحبا، كيف حالك اليوم؟",               "🇸🇦 Arabic",       "🇫🇷 French",       True,  False],
                ["人工知能は世界を変えています",             "🇯🇵 Japanese",    "🇬🇧 English (UK)", True,  False],
                ["Bonjour! Comment ça va?",              "🇫🇷 French",       "🇩🇪 German",       True,  False],
                ["¿Cuál es tu nombre?",                  "🇪🇸 Spanish",      "🇮🇹 Italian",      True,  False],
                ["Dreams don't work unless you do.",     "🇬🇧 English (UK)", "🇵🇰 Urdu",         True,  False],
                ["Я люблю изучать новые языки",          "🇷🇺 Russian",      "🇬🇧 English (UK)", True,  False],
                ["나는 오늘 매우 행복합니다",               "🇰🇷 Korean",      "🇬🇧 English (UK)", True,  False],
            ],
            inputs=[input_text, src_lang, tgt_lang, auto_detect, slow_speech],
            label="✨ Click any row to try it!"
        )

    # ── Handlers ──────────────────────────────────────────────────────────
    def swap_langs(s, t):
        return t, s

    swap_btn.click(fn=swap_langs, inputs=[src_lang, tgt_lang], outputs=[src_lang, tgt_lang])
    clear_btn.click(
        fn=lambda: ("", None, "—", "—"),
        outputs=[input_text, audio_out, src_info_box, char_info_box]
    )
    translate_btn.click(
        fn=translate_and_speak,
        inputs=[input_text, src_lang, tgt_lang, auto_detect, slow_speech],
        outputs=[output_text, audio_out, src_info_box, char_info_box]
    )
    input_text.submit(
        fn=translate_and_speak,
        inputs=[input_text, src_lang, tgt_lang, auto_detect, slow_speech],
        outputs=[output_text, audio_out, src_info_box, char_info_box]
    )

print("✅ LinguaBird is ready to fly! 🦜")

/tmp/ipykernel_1578/346811934.py:336: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=CSS, title="LinguaBird 🦜") as demo:


✅ LinguaBird is ready to fly! 🦜


In [23]:
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://bde6e1d76c7934d329.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://bde6e1d76c7934d329.gradio.live
